# Summit 2026 HOL — Semantic Layer
## Module 02: Define Business Metrics with Semantic Views

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🛠️ Creates a **semantic view** that defines business metrics on top of your analytics tables
2. 🛠️ Defines **dimensions** (client attributes, regions, sectors, time)
3. 🛠️ Defines **metrics** (AUM, portfolio value, trade volume, unrealized PnL)
4. 🛠️ Adds **verified queries** so the agent learns correct SQL patterns
5. 🛠️ Creates a **Cortex Search service** for unstructured research notes

---

### Why Semantic Views Matter:

🔹 Without a semantic layer, AI agents guess at column meanings and relationships. With semantic views:
- The agent **knows** that `AUM` means Assets Under Management (not a column name to guess)
- The agent **knows** how to join CLIENTS to POSITIONS to TRADES
- The agent **knows** that revenue = AUM * fee rate, not just a raw column
- Verified queries teach the agent the *correct* SQL patterns for common questions

---

### Estimated Time: **25–30 minutes**

### Prerequisites:
- Module 01 complete (NEXUS_HOL database with analytics tables loaded)

## Step 1: Set Context

In [ ]:
%%sql -r SetContext
USE ROLE NEXUS_ADMIN;
USE DATABASE NEXUS_HOL;
USE SCHEMA SEMANTIC;
USE WAREHOUSE NEXUS_WH;

### SQL vs. UI: Two Ways to Build

📌 In this lab, we create the semantic view and Cortex Search service using **SQL** — this gives you full control over every attribute, relationship, and metric, and makes the configuration reproducible and version-controllable.

However, you can also create both using the **Snowsight UI**:

| Object | UI Path | Feature |
|--------|---------|---------|
| **Semantic View** | AI & ML → Cortex AI → Analyst → **Create with Autopilot** | Auto-generates dimensions, metrics, and relationships from your tables |
| **Cortex Search Service** | AI & ML → Cortex AI → Search → **Create** | Guided wizard for selecting source table, search column, and attributes |

**[Autopilot](https://docs.snowflake.com/en/user-guide/views-semantic/autopilot)** is especially useful for getting started quickly — it analyzes your tables and proposes a semantic view that you can then refine. For this lab, we use SQL so you understand every component being configured.

> **Documentation:** [Semantic Views](https://docs.snowflake.com/en/user-guide/views-semantic/overview) | [Cortex Search](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
>
> **Quickstart:** [Getting Started with Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst#label-analyst-access-example)

## Step 2: Create the Nexus Capital Semantic View

### This semantic view defines:
- **3 logical tables**: CLIENTS, POSITIONS, TRADES
- **Relationships** between them (CLIENT_ID joins)
- **Dimensions**: client attributes, time, sectors, regions
- **Metrics**: AUM, portfolio value, unrealized PnL, trade volume, trade count
- **Verified queries**: Teach the agent correct SQL for common questions
- **SQL generation instructions**: Business rules the agent must follow

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE SEMANTIC VIEW NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV

  TABLES (
    clients AS NEXUS_HOL.ANALYTICS.CLIENTS
      PRIMARY KEY (CLIENT_ID)
      COMMENT = 'Client master data including institutional and HNW accounts with AUM and risk profiles',

    positions AS NEXUS_HOL.ANALYTICS.POSITIONS
      PRIMARY KEY (POSITION_ID)
      COMMENT = 'Current portfolio positions by client and symbol with market values and unrealized PnL',

    trades AS NEXUS_HOL.ANALYTICS.TRADES
      PRIMARY KEY (TRADE_ID)
      COMMENT = 'Trade execution history including buy/sell orders with prices, quantities, and status'
  )

  RELATIONSHIPS (
  -- ╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
  -- ║  XXX #1: Replace XXX with the column that joins these tables to clients.                                        ║
  -- ║  Hint: Look at the PRIMARY KEY defined on the clients table above. What column would a foreign key reference?   ║
  -- ╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
    positions_to_clients AS
      positions(XXX) REFERENCES clients,
    trades_to_clients AS
      trades(XXX) REFERENCES clients
  )

  FACTS (
    clients.AUM AS AUM
      COMMENT = 'Assets Under Management in USD. Total value of assets the client has with Nexus Capital.',
    positions.POSITION_QUANTITY AS QUANTITY
      COMMENT = 'Number of shares held in the position',
    positions.AVG_COST AS AVG_COST
      COMMENT = 'Average cost basis per share for the position',
    positions.CURRENT_PRICE AS CURRENT_PRICE
      COMMENT = 'Current market price per share',
    positions.MARKET_VALUE AS MARKET_VALUE
      COMMENT = 'Current market value of the position (quantity * current_price)',
    positions.UNREALIZED_PNL AS UNREALIZED_PNL
      COMMENT = 'Unrealized profit or loss on the position (market_value - cost_basis). Positive = gain, negative = loss.',
    trades.TRADE_QUANTITY AS QUANTITY
      COMMENT = 'Number of shares in the trade order',
    trades.TRADE_PRICE AS PRICE
      COMMENT = 'Execution price per share for the trade'
  )

  DIMENSIONS (
    clients.CLIENT_NAME AS CLIENT_NAME
      COMMENT = 'Full name of the client account',
    clients.ACCOUNT_TYPE AS ACCOUNT_TYPE
      COMMENT = 'Account classification: INSTITUTIONAL, HNW (High Net Worth), or RETAIL',
    clients.REGION AS REGION
      COMMENT = 'Geographic region: North America, EMEA, or APJ',
    clients.RISK_PROFILE AS RISK_PROFILE
      COMMENT = 'Investment risk tolerance: CONSERVATIVE, MODERATE, or AGGRESSIVE',
    clients.RELATIONSHIP_MANAGER AS RELATIONSHIP_MANAGER
      COMMENT = 'Name of the assigned relationship manager',
    clients.STATUS AS STATUS
      COMMENT = 'Client account status: ACTIVE or INACTIVE',
    clients.ONBOARDED_DATE AS ONBOARDED_DATE
      COMMENT = 'Date the client was onboarded',

    positions.SYMBOL AS SYMBOL
      COMMENT = 'Stock ticker symbol (e.g., AAPL, NVDA, MSFT)',
    positions.SECTOR AS SECTOR
      COMMENT = 'Market sector classification (Technology, Financials, Energy, Healthcare, etc.)',

    trades.TRADE_TYPE AS TRADE_TYPE
      COMMENT = 'Direction of the trade: BUY or SELL',
    trades.TRADE_STATUS AS STATUS
      COMMENT = 'Trade execution status: EXECUTED or SETTLED',
    trades.EXCHANGE AS EXCHANGE
      COMMENT = 'Exchange where trade was executed (NYSE, NASDAQ, OTC)',
    trades.TRADE_DATE AS TRADE_DATE
      COMMENT = 'Timestamp when the trade was executed',
    trades.TRADE_NOTES AS NOTES
      COMMENT = 'Trader notes explaining the rationale for the trade'
  )

  METRICS (
    clients.TOTAL_AUM AS SUM(clients.AUM)
      COMMENT = 'Total Assets Under Management across all clients in USD',

    clients.CLIENT_COUNT AS COUNT(DISTINCT CLIENT_ID)
      COMMENT = 'Number of distinct client accounts',

    positions.TOTAL_PORTFOLIO_VALUE AS SUM(positions.MARKET_VALUE)
      COMMENT = 'Total current market value across all positions in USD',

    positions.TOTAL_UNREALIZED_PNL AS SUM(positions.UNREALIZED_PNL)
      COMMENT = 'Total unrealized profit/loss across all positions. Positive = overall portfolio gains.',

    positions.POSITION_COUNT AS COUNT(DISTINCT POSITION_ID)
      COMMENT = 'Number of distinct portfolio positions',

    positions.AVG_POSITION_VALUE AS AVG(positions.MARKET_VALUE)
      COMMENT = 'Average market value per position',

    trades.TRADE_COUNT AS COUNT(DISTINCT TRADE_ID)
      COMMENT = 'Number of trade executions',

    trades.TOTAL_TRADE_VOLUME AS SUM(trades.TRADE_QUANTITY * trades.TRADE_PRICE)
      COMMENT = 'Total dollar volume of trades (quantity * price summed across all trades)'
  )

  COMMENT = 'Nexus Capital - Financial analytics semantic view covering clients, positions, and trades'

  AI_SQL_GENERATION '
    -- Business rules for SQL generation:
    -- 1. When asked about "top clients" or "biggest clients", rank by AUM unless otherwise specified.
    -- 2. When asked about portfolio performance, use UNREALIZED_PNL. Positive = gains.
    -- 3. Default time filter: if no date specified, include all available data.
    -- 4. "Active clients" means STATUS = ''ACTIVE''.
    -- 5. When asked about "recent trades", order by TRADE_DATE DESC and limit to last 7 days unless specified.
    -- 6. Trade volume = QUANTITY * PRICE. Always compute as dollar volume, not share count.
    -- 7. For region breakdowns, use the CLIENTS.REGION column (North America, EMEA, APJ).
    -- 8. When computing concentration, use MARKET_VALUE / total portfolio MARKET_VALUE.
  '

  AI_VERIFIED_QUERIES (
    top_clients_by_aum AS (
    -- ╔════════════════════════════════════════════════════════════════════════════════════════════════════════╗
    -- ║  XXX #2: Replace XXX with the correct status value.                                                    ║
    -- ║  Hint: We only want current clients. Check the DIMENSIONS section — STATUS can be ACTIVE or INACTIVE.  ║
    -- ╚════════════════════════════════════════════════════════════════════════════════════════════════════════╝
      QUESTION 'What are our top 5 clients by AUM?'
      SQL 'SELECT CLIENT_NAME, ACCOUNT_TYPE, REGION, AUM, RISK_PROFILE
      FROM NEXUS_HOL.ANALYTICS.CLIENTS
      WHERE STATUS = ''XXX''
      ORDER BY AUM DESC
      LIMIT 5'
    ),

    portfolio_value_by_sector AS (
      QUESTION 'What is the total portfolio value by sector?'
      SQL 'SELECT p.SECTOR, SUM(p.MARKET_VALUE) AS TOTAL_VALUE, SUM(p.UNREALIZED_PNL) AS TOTAL_PNL
      FROM NEXUS_HOL.ANALYTICS.POSITIONS p
      GROUP BY p.SECTOR
      ORDER BY TOTAL_VALUE DESC'
    ),

    recent_large_buy_trades AS (
      QUESTION 'Show me recent buy trades over $1M in value'
      SQL 'SELECT c.CLIENT_NAME, t.SYMBOL, t.QUANTITY, t.PRICE, (t.QUANTITY * t.PRICE) AS TRADE_VALUE, t.TRADE_DATE, t.NOTES
      FROM NEXUS_HOL.ANALYTICS.TRADES t
      JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON t.CLIENT_ID = c.CLIENT_ID
      WHERE t.TRADE_TYPE = ''BUY'' AND (t.QUANTITY * t.PRICE) > 1000000
      ORDER BY t.TRADE_DATE DESC'
    ),

    clients_highest_unrealized_gains AS (
      QUESTION 'Which clients have the highest unrealized gains?'
      SQL 'SELECT c.CLIENT_NAME, c.ACCOUNT_TYPE, SUM(p.UNREALIZED_PNL) AS TOTAL_UNREALIZED_PNL, SUM(p.MARKET_VALUE) AS TOTAL_PORTFOLIO_VALUE
      FROM NEXUS_HOL.ANALYTICS.CLIENTS c
      JOIN NEXUS_HOL.ANALYTICS.POSITIONS p ON c.CLIENT_ID = p.CLIENT_ID
      GROUP BY c.CLIENT_NAME, c.ACCOUNT_TYPE
      ORDER BY TOTAL_UNREALIZED_PNL DESC'
    ),

    aum_breakdown_by_region AS (
      QUESTION 'What is our AUM breakdown by region?'
      SQL 'SELECT REGION, COUNT(*) AS CLIENT_COUNT, SUM(AUM) AS TOTAL_AUM, AVG(AUM) AS AVG_AUM
      FROM NEXUS_HOL.ANALYTICS.CLIENTS
      WHERE STATUS = ''ACTIVE''
      GROUP BY REGION
      ORDER BY TOTAL_AUM DESC'
    ),

    tech_sector_exposure AS (
      QUESTION 'Show me the technology sector exposure across all clients'
      SQL 'SELECT c.CLIENT_NAME, p.SYMBOL, p.QUANTITY, p.MARKET_VALUE, p.UNREALIZED_PNL
      FROM NEXUS_HOL.ANALYTICS.POSITIONS p
      JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
      WHERE p.SECTOR = ''Technology''
      ORDER BY p.MARKET_VALUE DESC'
    )
  );

SELECT 'Semantic view NEXUS_CAPITAL_SV created successfully' AS STATUS;

## Step 3: Validate the Semantic View

### Confirm the semantic view was created and inspect its structure

In [ ]:
%%sql -r ShowSemanticView
-- Verify the semantic view exists
SHOW SEMANTIC VIEWS IN SCHEMA NEXUS_HOL.SEMANTIC;

In [ ]:
%%sql -r dataframe_13
-- Inspect dimensions
SHOW SEMANTIC DIMENSIONS IN NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV;

In [ ]:
%%sql -r InspectMetrics
-- Inspect metrics
SHOW SEMANTIC METRICS IN NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV;

In [ ]:
%%sql -r InspectFacts
-- Inspect facts
SHOW SEMANTIC FACTS IN NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV;

## 🧠 Going Deeper: Ontology & Knowledge Graphs on Snowflake

The semantic view you just built defines meaning for a **single domain** (Nexus Capital). But what happens when your enterprise has hundreds of interconnected domains — and the AI needs to reason across them?

**Ontology on Snowflake** takes the semantic layer concept further by adding:
- A formal **ontology** (shared vocabulary + business logic across domains)
- A **knowledge graph** (entity relationships that span systems)
- **Agent-based reasoning** over the graph for multi-hop questions

This pattern is especially relevant when customers say: *"Our AI gives inconsistent answers depending on which team asks"* or *"We need explainable AI that can trace its reasoning."*

**Not required for this lab** — but if you're working on CoWork/Cortex Agents use cases, this is a pattern worth understanding.

| Resource | Link |
|----------|------|
| Recording (60 min — Value + Positioning + Live Demo) | [Watch in Scout](https://scout.snowflake.com/learn/courses/50217/ontology-on-snowflake-enabling-business-aware-ai) |
| Field Positioning Guide | [Compass](https://snowflake.seismic.com/Link/Content/DCVQD6hM79dqQ8cWVhhCpdjQdXC3) |
| Slides | [Compass](https://snowflake.seismic.com/Link/Content/DCRJdjmTJ3FCmGhQJ2XQWBWpmddB) |
| FAQ (AE + SE Q&A) | [Compass](https://snowflake.seismic.com/Link/Content/DC2R3q3TQFc678CBq34jq9c99bM3) |
| Ontology Stack Builder (CoCo Skill) |[GitHub](https://github.com/Snowflake-Labs/cortex-code-skills/tree/main/skills/ontology-stack-builder) |
| Ontology on Snowflake (Full Repo) | [GitHub](https://github.com/Snowflake-Labs/ontology-on-snowflake) |

> **When to position Ontology:** Customer has inconsistent AI results across teams, needs explainable reasoning, or is comparing against Palantir/Neo4j for knowledge graph use cases.

---

Now let's continue building — next we'll create a Cortex Search service to give the agent access to unstructured research documents.

## 📐 Design Note: One Semantic View vs. Many

The `NEXUS_CAPITAL_SV` you just built covers one domain (clients, positions, trades — all related, all joined). In real customer deployments, the question of **how many semantic views to create** comes up quickly.

**Key constraint to know:** A single Cortex Analyst call operates against **one semantic view**. The agent cannot generate a SQL JOIN across two separate SVs — but it *can* call them sequentially and synthesize results.

| Situation | Recommendation |
|---|---|
| Tables naturally join within one domain | One SV — keep it together |
| ~100+ columns | Split by domain; accuracy degrades above the soft limit |
| Distinct business domains (Sales, HR, Finance) | Separate SVs + configure agent with one Cortex Analyst tool per SV |
| Cross-domain SQL JOIN required at query time | Unified SV with RELATIONSHIPS, or Composable SVs (May 2026) |

**The `NEXUS_CAPITAL_SV` is a good example of the right scope:** CLIENTS, POSITIONS, and TRADES are tightly related, all join on `CLIENT_ID`, and user questions naturally span all three. This is the ideal single-SV shape.

If Nexus Capital later added a separate HR or Compliance domain with its own tables and owners, those would warrant their own SVs — and the agent would be configured with multiple Cortex Analyst tools (one per SV) to handle cross-domain questions through multi-hop orchestration.

> **Field note:** Customers sometimes try to put everything in one giant SV. The ~100-column soft limit is a useful forcing function — it encourages domain decomposition that also makes the agent more accurate.

## Step 4: Create Cortex Search Service

### Build a search service over the research notes for RAG (Retrieval Augmented Generation)

This allows the agent to search through analyst research, risk assessments, and compliance reports when answering questions that require unstructured context.

> Optional resource: [**Cortex Search Overview (May 2026)**](https://snowflake.seismic.com/Link/Content/DCHqGW99mPhBXGcHQjBVQCbbCXbG)
> Covers Cortex Search architecture, hybrid retrieval, agent integration, Analytical Search, and CoWork integration.

In [ ]:
%%sql -r CreateCortexSearchService
CREATE OR REPLACE CORTEX SEARCH SERVICE NEXUS_HOL.AGENTS.NEXUS_RESEARCH_SEARCH
 -- ╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
 -- ║  XXX #3: Replace XXX with the column to index for search.                                                    ║
 -- ║  Hint: Which column in RESEARCH_NOTES contains the main text body? Look at the table schema from Module 01.  ║
 -- ╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
  ON XXX
  ATTRIBUTES TITLE, AUTHOR, CATEGORY, REGION, SYMBOLS_MENTIONED
  WAREHOUSE = NEXUS_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Search service over Nexus Capital research notes, market commentary, and compliance reports'
AS (
  SELECT
    NOTE_ID,
    TITLE,
    CONTENT,
    AUTHOR,
    CATEGORY,
    REGION,
    SYMBOLS_MENTIONED,
    PUBLISHED_DATE
  FROM NEXUS_HOL.ANALYTICS.RESEARCH_NOTES
);

SELECT 'Cortex Search service NEXUS_RESEARCH_SEARCH created' AS STATUS;

In [ ]:
%%sql -r ShowCortexSearchService
-- Verify the search service
SHOW CORTEX SEARCH SERVICES IN SCHEMA NEXUS_HOL.AGENTS;

## Step 5: Test the Semantic View with Cortex Analyst

### Verify the semantic view works by asking a natural language question

This uses Cortex Analyst directly to test that the semantic view generates correct SQL.

In [ ]:
%%sql -r dataframe_15
-- Test: Ask a question against the semantic view
-- This verifies that Cortex Analyst can interpret the semantic view correctly
SELECT * FROM SEMANTIC_VIEW(
  NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV
  DIMENSIONS CLIENTS.CLIENT_NAME
  METRICS POSITIONS.TOTAL_PORTFOLIO_VALUE
)
ORDER BY TOTAL_PORTFOLIO_VALUE DESC
LIMIT 3;

In [ ]:
%%sql -r dataframe_16
-- Test: Regional AUM breakdown
SELECT * FROM SEMANTIC_VIEW(
  NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV
  DIMENSIONS CLIENTS.REGION
  METRICS CLIENTS.TOTAL_AUM
)
ORDER BY TOTAL_AUM DESC;

## Step 6: Test Cortex Search

### Verify the search service can find relevant research notes

In [ ]:
%%sql -r TestSearchService
-- Test search: Find research about technology sector
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'NEXUS_HOL.AGENTS.NEXUS_RESEARCH_SEARCH',
    '{
      "query": "What is the outlook for technology stocks and AI infrastructure?",
      "columns": ["TITLE", "CONTENT", "AUTHOR", "CATEGORY"],
      "limit": 3
    }'
  )
) AS SEARCH_RESULTS;

In [ ]:
%%sql -r dataframe_17
-- Test search: Find compliance-related documents
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'NEXUS_HOL.AGENTS.NEXUS_RESEARCH_SEARCH',
    '{
      "query": "Are there any compliance issues or concentration risk warnings?",
      "columns": ["TITLE", "CONTENT", "AUTHOR", "CATEGORY"],
      "limit": 3
    }'
  )
) AS SEARCH_RESULTS;

## ✅ Module 02 Complete!

### You now have:
- Semantic view: `NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV`
  - 3 logical tables with defined relationships
  - 15 dimensions (client attributes, sectors, trade details)
  - 8 facts (prices, quantities, values)
  - 8 metrics (AUM, portfolio value, PnL, trade volume, counts)
  - 6 verified queries (teach the agent correct patterns)
  - SQL generation instructions (business rules)
- Cortex Search service: `NEXUS_HOL.AGENTS.NEXUS_RESEARCH_SEARCH`
  - Indexes 8 research documents (notes, commentary, compliance)
  - Searchable by content with attribute filtering

---

### What You Just Built:

The **semantic layer** is what transforms raw analytics tables into *meaning*. Without it, an AI agent would:
- Guess at column meanings
- Write incorrect joins
- Misinterpret business metrics
- Generate SQL that returns wrong answers

With it, the agent has a governed, validated understanding of your business — and verified queries that prove it works.

---

### Key Talking Points:

- "The semantic layer is how you give AI *understanding*, not just *access*." 
- "Verified queries are like unit tests for your AI — they ensure the agent generates correct SQL for known questions."
- "Cortex Search gives the agent unstructured context (research, compliance) alongside structured analytics."

---

### Next Steps:

Continue to **Module 03: Build the Cortex Agent (API)** — Wire the semantic view and search service into a Cortex Agent and interact with it via the REST API.